# Sequential Workflow: Automated ML Assignment Review

This notebook demonstrates a deterministic sequential workflow using `picoagents.workflow`.

Scenario:

In the course Introduction to Machine Learning at Gisma University, a student submits an HTML-exported Jupyter notebook for an ML assignment.

The review pipeline follows a fixed sequence:

1. Check notebook structure
2. Run basic code checks
3. Evaluate model-performance evidence
4. Generate feedback summary

This is a sequential workflow because each step depends on the previous one.

## Setup

In [8]:
import os
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel
from typing import Optional, Literal, List, Dict, Any

from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root or parent directory.
# Adjust this path if your notebook is located elsewhere.
load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    print("API key loaded successfully.")
else:
    print("OPENAI_API_KEY is empty. Deterministic workflow examples can still run, but LLM-agent examples need an API key.")

client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)



from picoagents.workflow import Workflow, WorkflowRunner, FunctionStep
from picoagents.workflow.core import WorkflowMetadata, StepMetadata, Context

runner = WorkflowRunner()

API key loaded successfully.


## Define typed input and output models

In [9]:
class SubmissionInput(BaseModel):
    """Input contract for one student submission to be reviewed."""
    student_id: str
    notebook_title: str
    has_sections: bool
    code_runs: bool
    includes_two_models: bool
    includes_hyperparameter_search: bool


class StructureCheckOutput(BaseModel):
    """Result of validating required notebook structure fields."""
    student_id: str
    notebook_title: str
    structure_ok: bool
    code_runs: bool
    includes_two_models: bool
    includes_hyperparameter_search: bool
    notes: List[str]


class CodeCheckOutput(BaseModel):
    """Result of basic code execution checks plus carried context."""
    student_id: str
    notebook_title: str
    structure_ok: bool
    code_ok: bool
    includes_two_models: bool
    includes_hyperparameter_search: bool
    notes: List[str]


class PerformanceReviewOutput(BaseModel):
    """Scored review summary for model-quality evidence."""
    student_id: str
    notebook_title: str
    score: int
    notes: List[str]


class FeedbackOutput(BaseModel):
    """Final human-readable feedback produced by the workflow."""
    result: str

## Define workflow step functions

In [10]:
async def check_notebook_structure(input_data: SubmissionInput, context: Context) -> StructureCheckOutput:
    """Verify required notebook sections and capture structure notes."""
    notes = []
    if input_data.has_sections:
        notes.append("Notebook has the expected assignment sections.")
    else:
        notes.append("Missing one or more required sections.")
    return StructureCheckOutput(
        student_id=input_data.student_id,
        notebook_title=input_data.notebook_title,
        structure_ok=input_data.has_sections,
        code_runs=input_data.code_runs,
        includes_two_models=input_data.includes_two_models,
        includes_hyperparameter_search=input_data.includes_hyperparameter_search,
        notes=notes
    )


async def run_basic_code_checks(input_data: StructureCheckOutput, context: Context) -> CodeCheckOutput:
    """Mark whether code runs and extend notes from prior steps."""
    notes = list(input_data.notes)
    if input_data.code_runs:
        notes.append("Code execution check passed.")
    else:
        notes.append("Code execution check failed.")
    return CodeCheckOutput(
        student_id=input_data.student_id,
        notebook_title=input_data.notebook_title,
        structure_ok=input_data.structure_ok,
        code_ok=input_data.code_runs,
        includes_two_models=input_data.includes_two_models,
        includes_hyperparameter_search=input_data.includes_hyperparameter_search,
        notes=notes
    )


async def evaluate_model_evidence(input_data: CodeCheckOutput, context: Context) -> PerformanceReviewOutput:
    """Compute a simple rubric score from structure, code, and modeling evidence."""
    notes = list(input_data.notes)
    score = 0

    if input_data.structure_ok:
        score += 25
    if input_data.code_ok:
        score += 25
    if input_data.includes_two_models:
        score += 25
        notes.append("The submission compares at least two model families.")
    else:
        notes.append("The submission should compare at least two model families.")
    if input_data.includes_hyperparameter_search:
        score += 25
        notes.append("The submission includes hyperparameter exploration.")
    else:
        notes.append("Hyperparameter exploration is missing or too shallow.")

    return PerformanceReviewOutput(
        student_id=input_data.student_id,
        notebook_title=input_data.notebook_title,
        score=score,
        notes=notes
    )


async def generate_feedback_report(input_data: PerformanceReviewOutput, context: Context) -> FeedbackOutput:
    """Format the final score and notes into a readable feedback report."""
    feedback = (
        f"Student: {input_data.student_id}\n"
        f"Notebook: {input_data.notebook_title}\n"
        f"Score: {input_data.score}/100\n\n"
        "Feedback:\n- " + "\n- ".join(input_data.notes)
    )
    return FeedbackOutput(result=feedback)

## Wrap functions as workflow steps

In [11]:
structure_step = FunctionStep(
    step_id="check_structure",
    metadata=StepMetadata(name="Check Notebook Structure"),
    input_type=SubmissionInput,
    output_type=StructureCheckOutput,
    func=check_notebook_structure
)

code_step = FunctionStep(
    step_id="run_code_checks",
    metadata=StepMetadata(name="Run Basic Code Checks"),
    input_type=StructureCheckOutput,
    output_type=CodeCheckOutput,
    func=run_basic_code_checks
)

performance_step = FunctionStep(
    step_id="evaluate_model_evidence",
    metadata=StepMetadata(name="Evaluate Model Evidence"),
    input_type=CodeCheckOutput,
    output_type=PerformanceReviewOutput,
    func=evaluate_model_evidence
)

feedback_step = FunctionStep(
    step_id="generate_feedback",
    metadata=StepMetadata(name="Generate Feedback Report"),
    input_type=PerformanceReviewOutput,
    output_type=FeedbackOutput,
    func=generate_feedback_report
)

## Compose and run the sequential workflow

In [12]:
sequential_workflow = (
    Workflow(metadata=WorkflowMetadata(name="Sequential ML Assignment Review"))
    .chain(structure_step, code_step, performance_step, feedback_step)
)

sample_submission = {
    "student_id": "S1024",
    "notebook_title": "Scikit-learn ML Pipeline Assignment",
    "has_sections": True,
    "code_runs": True,
    "includes_two_models": True,
    "includes_hyperparameter_search": False
}

async for event in runner.run_stream(sequential_workflow, sample_submission):
    print(event)

[19:46:50] 🚀 Workflow started with input: {'student_id': 'S1024', 'notebook_title': 'Scikit-learn ML Pipeline Assignment', 'has_sections': True, 'code_runs': True, 'includes_two_models': True, 'includes_hyperparameter_search': False}
[19:46:50] ▶️  Step 'check_structure' started
[19:46:50] ✅ Step 'check_structure' completed → {'student_id': 'S1024', 'notebook_title': 'Scikit-learn ML Pipeline Assignment', 'structure_ok': True, 'code_runs': True, 'includes_two_models': True, 'includes_hyperparameter_search': False, 'notes': ['Notebook has the expected assignment sections.']}
[19:46:50] 🔗 check_structure → run_code_checks
[19:46:50] ▶️  Step 'run_code_checks' started
[19:46:50] ✅ Step 'run_code_checks' completed → {'student_id': 'S1024', 'notebook_title': 'Scikit-learn ML Pipeline Assignment', 'structure_ok': True, 'code_ok': True, 'includes_two_models': True, 'includes_hyperparameter_search': False, 'notes': ['Notebook has the expected assignment sections.', 'Code execution check passed

## Reflection questions

1. Which step would fail first if the input data had a wrong type?
2. Why is this pattern easy to debug?
3. Which part of this example could be replaced by an LLM agent later?

## Student Exercises

Task 1: Add a new deterministic step called check_references between run_basic_code_checks and evaluate_model_evidence.


Task 2: Change the scoring rubric so includes_hyperparameter_search contributes 15 points and code_ok contributes 35 points.


Task 3: Extend FeedbackOutput with a pass_fail field and populate it based on score >= 70.

## Solutions

Solution 1

```python
class ReferenceCheckOutput(BaseModel):
    student_id: str
    notebook_title: str
    structure_ok: bool
    code_ok: bool
    references_ok: bool
    includes_two_models: bool
    includes_hyperparameter_search: bool
    notes: List[str]

async def check_references(input_data: CodeCheckOutput, context: Context) -> ReferenceCheckOutput:
    notes = list(input_data.notes)
    references_ok = "reference" in input_data.notebook_title.lower() or True
    notes.append("Reference check completed.")
    return ReferenceCheckOutput(
        student_id=input_data.student_id,
        notebook_title=input_data.notebook_title,
        structure_ok=input_data.structure_ok,
        code_ok=input_data.code_ok,
        references_ok=references_ok,
        includes_two_models=input_data.includes_two_models,
        includes_hyperparameter_search=input_data.includes_hyperparameter_search,
        notes=notes,
    )

# Update chain: structure -> code -> references -> performance -> feedback
```

Solution 2

```python
if input_data.code_ok:
    score += 35
if input_data.includes_hyperparameter_search:
    score += 15
```

Solution 3

```python
class FeedbackOutput(BaseModel):
    result: str
    pass_fail: str

pass_fail = "PASS" if input_data.score >= 70 else "FAIL"
return FeedbackOutput(result=feedback, pass_fail=pass_fail)
```